In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns; sns.set()
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import LeaveOneOut, KFold, train_test_split
from sklearn.metrics import accuracy_score
import random
from timeit import default_timer as timer
from ISLP import load_data

In [3]:
df=load_data('Default')
df.head(10)

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879
5,No,Yes,919.588530,7491.558572
6,No,No,825.513331,24905.226578
7,No,Yes,808.667504,17600.451344
8,No,No,1161.057854,37468.529288
9,No,No,0.000000,29275.268293


In [4]:
# Note: factorize() returns two objects: a label array and an array with the unique values.
# We are only interested in the first object. 
df['default2'] = df.default.factorize()[0]
df['student2'] = df.student.factorize()[0]
df.head(3)

,default,student,balance,income,default2,student2
0,No,No,729.526495,44361.625074,0,0
1,No,Yes,817.180407,12106.134700,0,1
2,No,No,1073.549164,31767.138947,0,0


In [5]:
X = df[['balance', 'income', 'student2']]
y=df.default2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

In [6]:
class ClassifierEvaluator:
    def __init__(self, clf, X, y):
        self.clf = clf
        self.X = X
        self.y = y
        
    def evaluate_loocv(self):
        """Leave One Out Cross-Validation"""
        start= timer()
        loocv = LeaveOneOut()
        scores = []
        for train_index, test_index in loocv.split(self.X):
            X_train, X_test = self.X.iloc[train_index], self.X.iloc[test_index]
            y_train, y_test = self.y.iloc[train_index], self.y.iloc[test_index]
            self.clf.fit(X_train, y_train)
            y_pred = self.clf.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            scores.append(score)
        end = timer()
        return np.mean(scores), end-start
    
    def evaluate_kfold(self, k=5):
        """K-fold Cross-Validation"""
        kf = KFold(n_splits=k)
        scores = []
        for train_index, test_index in kf.split(self.X):
            X_train, X_test = self.X.iloc[train_index], self.X.iloc[test_index]
            y_train, y_test = self.y.iloc[train_index], self.y.iloc[test_index]
            self.clf.fit(X_train, y_train)
            y_pred = self.clf.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            scores.append(score)
        return np.mean(scores)
    
    def evaluate_validation_set(self, test_size=0.2):
        """Validation Set Approach"""
        X_train, X_val, y_train, y_val = train_test_split(self.X, self.y, test_size=test_size)
        self.clf.fit(X_train, y_train)
        y_pred = self.clf.predict(X_val)
        return accuracy_score(y_val, y_pred)
    
    def evaluate_bootstrap(self, n_samples=1000):
        """Bootstrap Resampling"""
        scores = []
        for _ in range(n_samples):
            # Draw a random sample from the data with replacement
            sample_index = np.random.choice(self.X.index, len(self.X), replace=True)
            X_sample, y_sample = self.X.loc[sample_index], self.y.loc[sample_index]
            self.clf.fit(X_sample, y_sample)
            y_pred = self.clf.predict(X_sample)
            score = accuracy_score(y_sample, y_pred)
            scores.append(score)
        return np.mean(scores)

In [ ]:
clf = LogisticRegression()
evaluator = ClassifierEvaluator(clf, X, y)
# call the evaluate_loocv() method to get the LOOCV score
loocv_score, time = evaluator.evaluate_loocv()
print("LOOCV score: ", loocv_score)
print("Time taken by the function to compute the LOOCV is", time)

# call the evaluate_kfold() method to get the K-fold CV score
kfold_score = evaluator.evaluate_kfold()
print("K-fold CV score: ", kfold_score)

# call the evaluate_validation_set() method to get the validation set score
validation_set_score = evaluator.evaluate_validation_set()
print("Validation set score: ", validation_set_score)

# call the evaluate_bootstrap() method to get the bootstrap resampling score
bootstrap_score = evaluator.evaluate_bootstrap()
print("Bootstrap score: ", bootstrap_score)

/Users/ssehra/miniconda3/envs/mlcourse/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/ssehra/miniconda3/envs/mlcourse/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.or